In [ ]:
# 02_B_DASCH_Replica.ipynb - Setup
# Uses real SExtractor + astrometry.net (no Windows builds - runs in WSL2)

# Step 0 : Install WSL2 (PowerShell, as Administrator)
# wsl --install -d Ubuntu
# Restart, open "Ubuntu" from Start menu, set up username/password

# Step 1 : Install tools (Ubuntu terminal)
# sudo apt update
# sudo apt install source-extractor astrometry.net -y
# source-extractor --version
# solve-field --help

# Step 2 : Astrometry index files (Ubuntu terminal)
# sudo mkdir -p /usr/share/astrometry
# cd /usr/share/astrometry
# sudo rm -f index-41*.fits
# sudo wget https://portal.nersc.gov/project/cosmo/temp/dstn/index-5200/LITE/index-520{0..6}-42.fits
# sudo wget https://portal.nersc.gov/project/cosmo/temp/dstn/index-5200/LITE/index-520{0..6}-45.fits
# sudo wget https://portal.nersc.gov/project/cosmo/temp/dstn/index-5200/LITE/index-520{0..6}-46.fits
# grep -q "add_path /usr/share/astrometry" /etc/astrometry.cfg || echo "add_path /usr/share/astrometry" | sudo tee -a /etc/astrometry.cfg
# cat /etc/astrometry.cfg

# Step 3 : Python venv (Ubuntu terminal)
# cd ~
# python3 -m venv replica_dasch_venv
# sudo apt update
# sudo apt install -y build-essential libcairo2-dev pkg-config python3-dev
# source ~/replica_dasch_venv/bin/activate
# pip install numpy pandas matplotlib astropy photutils scipy jupyter ipywidgets ipykernel seaborn scikit-learn scikit-image
# pip install daschlab

# Step 4 : Register kernel (Ubuntu terminal)
# python -m ipykernel install --user --name=replica_dasch --display-name="Python (replica_DASCH)"

# Step 5 : VS Code
# Install from code.visualstudio.com (genuine MS VS Code, not a fork - WSL extension is MS-only)

# Step 6 : Connect (VS Code)
# Ctrl+Shift+X -> install "WSL" (Microsoft)
# Ctrl+Shift+P -> "WSL: Connect to WSL" -> confirm green WSL: Ubuntu badge
# File -> Open Folder -> notebooks folder in Windows Explorer path, opened through the WSL file picker
# With WSL badge showing, install "Python" + "Jupyter" (Microsoft) again - separate from Windows-side extensions

# Step 7 : Run (VS Code)
# Open 02_B_DASCH_Replica.ipynb -> Select Kernel -> Python (replica_DASCH) -> run Cell 1

# Reset if needed :
# rm -rf ~/replica_dasch_venv
# jupyter kernelspec uninstall replica_dasch -y

In [ ]:
# NOTE : 02_B_DASCH_Replica.ipynb - Cell 1
# Tool verification + SExtractor config (run this before running cell 3)

import subprocess
import shutil
from pathlib import Path

# --- Verify tools ---
SEXTRACTOR_CMD = None
for candidate in ['source-extractor', 'sextractor']:
    try:
        subprocess.run([candidate, '--version'], capture_output=True, timeout=10)
        SEXTRACTOR_CMD = candidate
        print(f"✓ SExtractor found as '{candidate}'")
        break
    except FileNotFoundError:
        continue
if SEXTRACTOR_CMD is None:
    print("❌ SExtractor not found")

try:
    subprocess.run(['solve-field', '--help'], capture_output=True, timeout=10)
    print("✓ solve-field found")
except FileNotFoundError:
    print("❌ solve-field not found")

# --- Paths ---
BASE_DATA_DIR = Path("/mnt/c/Users/dapur/Downloads/Other/Research/rcb_Star_Dust_Survey/test_CNN/data/V_CrA")
ASTROMETRY_CONFIG = Path("/etc/astrometry.cfg")

print(f"\nData dir: {BASE_DATA_DIR}")
print(f"Exists: {BASE_DATA_DIR.exists()}")
print(f"Astrometry config exists: {ASTROMETRY_CONFIG.exists()}")

# --- SExtractor config (requesting MAG_ISO) ---
EXTRACTOR_CONFIG_DIR = Path.home() / "extractor_config"
EXTRACTOR_CONFIG_DIR.mkdir(exist_ok=True)

EXTRACTOR_PARAM_FILE = EXTRACTOR_CONFIG_DIR / "default.param"
EXTRACTOR_PARAM_FILE.write_text(
    "NUMBER\nX_IMAGE\nY_IMAGE\nALPHA_J2000\nDELTA_J2000\n"
    "MAG_ISO\nMAGERR_ISO\nMAG_AUTO\nMAGERR_AUTO\nFLUX_ISO\n"
    "FLAGS\nISOAREA_IMAGE\nELONGATION\nFWHM_IMAGE\n"
)

EXTRACTOR_CONFIG_FILE = EXTRACTOR_CONFIG_DIR / "default.extractor"
EXTRACTOR_CONFIG_FILE.write_text(f"""
CATALOG_NAME     out.cat
CATALOG_TYPE     ASCII_HEAD
PARAMETERS_NAME  {EXTRACTOR_PARAM_FILE}
DETECT_TYPE      CCD
DETECT_MINAREA   5
DETECT_THRESH    3.0
ANALYSIS_THRESH  3.0
FILTER           N
DEBLEND_NTHRESH  32
DEBLEND_MINCONT  0.005
CLEAN            Y
CLEAN_PARAM      1.0
MASK_TYPE        CORRECT
PHOT_APERTURES   10
PHOT_AUTOPARAMS  2.5, 3.5
SATUR_LEVEL      50000.0
MAG_ZEROPOINT    25.0
GAIN             1.0
PIXEL_SCALE      0
SEEING_FWHM      2.0
STARNNW_NAME     {EXTRACTOR_CONFIG_DIR}/default.nnw
BACK_SIZE        64
BACK_FILTERSIZE  3
VERBOSE_TYPE     QUIET
""")

# default.nnw - SExtractor's stock neural network file, must exist even if unused
nnw_path = EXTRACTOR_CONFIG_DIR / "default.nnw"
if not nnw_path.exists():
    result = subprocess.run(['find', '/usr', '/etc', '-name', 'default.nnw'],
                             capture_output=True, text=True, timeout=30)
    found = result.stdout.splitlines()
    if found:
        shutil.copy(found[0], nnw_path)
        print(f"✓ Copied default.nnw from {found[0]}")
    else:
        print("⚠ default.nnw not found automatically - locate it under the SExtractor "
              "install (e.g. /usr/share/source-extractor/) and copy it to:", nnw_path)

print(f"\nSExtractor config: {EXTRACTOR_CONFIG_FILE}")
print(f"SExtractor params: {EXTRACTOR_PARAM_FILE}")

✓ SExtractor found as 'source-extractor'
✓ solve-field found

Data dir: /mnt/c/Users/dapur/Downloads/Other/Research/rcb_Star_Dust_Survey/test_CNN/data/V_CrA
Exists: True
Astrometry config exists: True

SExtractor config: /home/dapur/extractor_config/default.extractor
SExtractor params: /home/dapur/extractor_config/default.param


In [ ]:
# NOTE 02_B_DASCH_Replica.ipynb - Cell 2
# Run on synthetic unit tests

import subprocess
import shelve
import shutil
import time
import tempfile
import traceback
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from astropy.io import fits as astrofits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from IPython.display import display
import ipywidgets as widgets

UNIT_TEST_DIR = Path("/mnt/c/Users/dapur/Downloads/Other/Research/rcb_Star_Dust_Survey/test_CNN/data/unit_Tests")
CUTOUTS_DIR = UNIT_TEST_DIR / 'cutouts'
CATALOG_PATH = UNIT_TEST_DIR / 'synthetic_catalog.csv'

RUN_TIMESTAMP = time.strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = UNIT_TEST_DIR / f'test_Algorithm_{RUN_TIMESTAMP}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = UNIT_TEST_DIR / 'algorithm_cache_02B'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / 'algorithm_photometry_results.csv'
LIGHTCURVE_PATH = OUTPUT_DIR / 'algorithm_lightcurve.csv'
REF_PHOTOMETRY_PATH = OUTPUT_DIR / 'algorithm_reference_photometry.csv'

print(f"Output folder: {OUTPUT_DIR}")

MATCH_RADIUS_PX = 5.0
ISOLATION_MIN_SEPARATION = 20
REF_STARS_MAX_PER_PLATE = 150
MIN_REF_STARS = 25
LOCAL_HALF_WINDOW = 1.5
MIN_REF_STARS_LOCAL = 8
LOCAL_WINDOW_GROWTH = 1.6
MIN_ABS_LOCAL_SLOPE = 0.3
MAX_CALIBRATION_RMS = 0.3

PIPELINE_VERSION = 3

catalog = pd.read_csv(CATALOG_PATH)
target_row = catalog[catalog['is_target'] == True].iloc[0]
TARGET_COORD = SkyCoord(ra=target_row['ra'] * u.deg, dec=target_row['dec'] * u.deg)
ref_catalog = catalog[(catalog['is_target'] == False) & (catalog['is_phantom'] == False)].copy()
print(f"Loaded catalog: {len(catalog)} stars ({len(ref_catalog)} non-phantom reference candidates)")

cutouts = sorted(CUTOUTS_DIR.glob('*.fits'))
print(f"Found {len(cutouts)} plates")

if CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
CACHE_DIR.mkdir(parents=True)
plate_db = shelve.open(str(CACHE_DIR / 'plate_db_shelf'), flag='c', writeback=False)
print(f"Cache cleared, starting fresh")

def run_sextractor(fits_path, workdir):
    workdir = Path(workdir)
    catalog_out = workdir / f"{fits_path.stem}.cat"
    cmd = [
        SEXTRACTOR_CMD, str(fits_path),
        '-c', str(EXTRACTOR_CONFIG_FILE),
        '-CATALOG_NAME', str(catalog_out),
        '-PARAMETERS_NAME', str(EXTRACTOR_PARAM_FILE),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if not catalog_out.exists():
        return None, result.stderr

    colnames = []
    with open(catalog_out) as f:
        for line in f:
            if line.startswith('#'):
                colnames.append(line.split()[2])
            else:
                break
    if not colnames:
        return None, "empty catalog"
    df = pd.read_csv(catalog_out, comment='#', sep=r'\s+', names=colnames)
    return df, None

def filter_isolated(x, y, min_sep=ISOLATION_MIN_SEPARATION):
    n = len(x)
    if n < 2:
        return np.ones(n, dtype=bool)
    isolated = np.ones(n, dtype=bool)
    for i in range(n):
        d = np.hypot(x - x[i], y - y[i])
        d[i] = np.inf
        if np.any(d < min_sep):
            isolated[i] = False
    return isolated

def calibrate_global(inst_mags, apass_b, min_stars=MIN_REF_STARS):
    mask = np.isfinite(inst_mags) & np.isfinite(apass_b)
    if mask.sum() < min_stars:
        return None
    x, y = apass_b[mask], inst_mags[mask]
    for _ in range(3):
        try:
            coeffs = np.polyfit(x, y, 2)
            residuals = y - np.polyval(coeffs, x)
            rms = np.sqrt(np.mean(residuals**2))
            good = np.abs(residuals) < 3.0 * rms
            if good.sum() < min_stars:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    if len(x) < min_stars:
        return None
    coeffs = np.polyfit(x, y, 2)
    residuals = y - np.polyval(coeffs, x)
    rms = np.sqrt(np.mean(residuals**2))
    return {'coeffs': coeffs, 'rms': rms, 'n_used': len(x),
            'b_min': float(np.min(x)), 'b_max': float(np.max(x))}

def calibrate_local(inst_mags, apass_b, target_b_estimate,
                     half_window=LOCAL_HALF_WINDOW, min_stars=MIN_REF_STARS_LOCAL,
                     growth=LOCAL_WINDOW_GROWTH):
    mask = np.isfinite(inst_mags) & np.isfinite(apass_b)
    x_all, y_all = apass_b[mask], inst_mags[mask]
    if len(x_all) < min_stars:
        return None
    window = half_window
    local_mask = np.abs(x_all - target_b_estimate) <= window
    for _ in range(5):
        if local_mask.sum() >= min_stars:
            break
        window *= growth
        local_mask = np.abs(x_all - target_b_estimate) <= window
    if local_mask.sum() < min_stars:
        return None
    x, y = x_all[local_mask], y_all[local_mask]
    for _ in range(3):
        try:
            coeffs = np.polyfit(x, y, 1)
            residuals = y - np.polyval(coeffs, x)
            rms = np.sqrt(np.mean(residuals**2))
            good = np.abs(residuals) < 3.0 * rms
            if good.sum() < min_stars:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    if len(x) < min_stars:
        return None
    coeffs = np.polyfit(x, y, 1)
    residuals = y - np.polyval(coeffs, x)
    rms = np.sqrt(np.mean(residuals**2))
    return {'coeffs': coeffs, 'rms': rms, 'n_used': len(x), 'window_used': float(window),
            'b_min': float(np.min(x)), 'b_max': float(np.max(x))}

def invert_calibration(calibration, t_inst_mag):
    coeffs = list(calibration['coeffs'])
    coeffs[-1] -= t_inst_mag
    roots = np.roots(coeffs)
    real_roots = roots[np.abs(roots.imag) < 1e-6].real
    if len(real_roots) == 0:
        return None
    lo, hi = calibration['b_min'], calibration['b_max']
    in_range = real_roots[(real_roots >= lo - 2) & (real_roots <= hi + 2)]
    candidates = in_range if len(in_range) > 0 else real_roots
    return float(min(candidates, key=lambda r: abs(np.polyval(calibration['coeffs'], r) - t_inst_mag)))

def process_plate(fits_path):
    try:
        plate_rng = np.random.default_rng(abs(hash(fits_path.name)) % (2**32))
        header = astrofits.getheader(fits_path)
        wcs = WCS(header)
        jd = header.get('JD-OBS', np.nan)
        h, w = astrofits.getdata(fits_path).shape

        with tempfile.TemporaryDirectory() as workdir:
            sources, err = run_sextractor(fits_path, workdir)
        if sources is None or len(sources) == 0:
            return _empty_result(fits_path, jd, f'SExtractor found nothing ({err})')

        det_x = sources['X_IMAGE'].values
        det_y = sources['Y_IMAGE'].values
        det_mag_iso = sources['MAG_ISO'].values
        det_flags = sources['FLAGS'].values.astype(int)
        is_saturated_flag = (det_flags & 4) != 0

        ra = ref_catalog['ra'].values
        dec = ref_catalog['dec'].values
        apass_b_all = ref_catalog['apass_b_mag'].values
        cat_x, cat_y = wcs.all_world2pix(ra, dec, 0)
        margin = 20
        in_frame = (cat_x > margin) & (cat_x < w - margin) & (cat_y > margin) & (cat_y < h - margin)
        cat_x, cat_y, apass_b_all = cat_x[in_frame], cat_y[in_frame], apass_b_all[in_frame]

        if len(cat_x) > REF_STARS_MAX_PER_PLATE:
            keep = plate_rng.choice(len(cat_x), size=REF_STARS_MAX_PER_PLATE, replace=False)
            cat_x, cat_y, apass_b_all = cat_x[keep], cat_y[keep], apass_b_all[keep]

        matched_inst_mag, matched_apass_b, matched_x, matched_y, matched_sat = [], [], [], [], []
        for cx, cy, ab in zip(cat_x, cat_y, apass_b_all):
            d = np.hypot(det_x - cx, det_y - cy)
            j = np.argmin(d)
            if d[j] <= MATCH_RADIUS_PX:
                matched_inst_mag.append(det_mag_iso[j])
                matched_apass_b.append(ab)
                matched_x.append(det_x[j])
                matched_y.append(det_y[j])
                matched_sat.append(is_saturated_flag[j])

        if len(matched_inst_mag) == 0:
            return _empty_result(fits_path, jd, 'No SExtractor detections matched catalog stars')

        matched_inst_mag = np.array(matched_inst_mag)
        matched_apass_b = np.array(matched_apass_b)
        matched_x = np.array(matched_x)
        matched_y = np.array(matched_y)
        matched_sat = np.array(matched_sat)

        isolated = filter_isolated(matched_x, matched_y)
        good_for_fit = isolated & ~matched_sat & np.isfinite(matched_inst_mag)

        calibration_global = calibrate_global(matched_inst_mag[good_for_fit], matched_apass_b[good_for_fit])
        n_ref_used = calibration_global['n_used'] if calibration_global else int(good_for_fit.sum())

        ref_rows = []
        for i in np.where(good_for_fit)[0]:
            ref_rows.append({'filename': fits_path.name, 'star_id': f'MATCH_{i:04d}',
                              'apass_b_mag': matched_apass_b[i], 'instrumental_mag': matched_inst_mag[i]})

        tx, ty = wcs.all_world2pix(TARGET_COORD.ra.deg, TARGET_COORD.dec.deg, 0)
        target_detected = False
        target_mag = np.nan
        target_mag_error = np.nan
        calibration_mode = None
        final_rms = np.nan
        final_n_used = 0

        if margin < tx < w - margin and margin < ty < h - margin and calibration_global is not None:
            d = np.hypot(det_x - tx, det_y - ty)
            j = np.argmin(d)
            if d[j] <= MATCH_RADIUS_PX:
                t_inst_mag = det_mag_iso[j]
                rough_mag = invert_calibration(calibration_global, t_inst_mag)
                if rough_mag is not None:
                    calibration_local = calibrate_local(matched_inst_mag[good_for_fit],
                                                          matched_apass_b[good_for_fit], rough_mag)
                    if (calibration_local is not None
                            and abs(calibration_local['coeffs'][0]) >= MIN_ABS_LOCAL_SLOPE):
                        local_mag = invert_calibration(calibration_local, t_inst_mag)
                        if local_mag is not None:
                            target_mag = local_mag
                            final_rms = calibration_local['rms']
                            final_n_used = calibration_local['n_used']
                            calibration_mode = 'local'
                    if calibration_mode is None:
                        target_mag = rough_mag
                        final_rms = calibration_global['rms']
                        final_n_used = calibration_global['n_used']
                        calibration_mode = 'global_fallback'
                    target_mag_error = final_rms / np.sqrt(max(final_n_used, 1))
                    target_detected = True

        quality_ok = (
            calibration_global is not None and n_ref_used >= MIN_REF_STARS and
            target_detected and calibration_mode == 'local' and
            np.isfinite(final_rms) and final_rms <= MAX_CALIBRATION_RMS
        )

        rejection_reason = None
        if calibration_global is None:
            rejection_reason = 'Global calibration failed'
        elif n_ref_used < MIN_REF_STARS:
            rejection_reason = f'Too few reference stars ({n_ref_used} < {MIN_REF_STARS})'
        elif not target_detected:
            rejection_reason = 'Target not detected/matched'
        elif calibration_mode != 'local':
            rejection_reason = 'Local calibration unavailable'
        elif final_rms > MAX_CALIBRATION_RMS:
            rejection_reason = f'RMS too high ({final_rms:.3f})'

        result_row = {
            'filename': fits_path.name, 'jd': jd, 'target_detected': target_detected,
            'target_mag': target_mag, 'target_mag_error': target_mag_error,
            'target_x': float(tx), 'target_y': float(ty),
            'num_reference_stars': n_ref_used,
            'zeropoint': calibration_global['coeffs'][-1] if calibration_global else np.nan,
            'rms_scatter': final_rms,
            'calibration_mode': calibration_mode, 'quality_ok': quality_ok,
            'rejection_reason': rejection_reason,
        }
        return {'result_row': result_row, 'ref_rows': ref_rows}

    except Exception as e:
        return _empty_result(fits_path, np.nan, str(e))

def _empty_result(fits_path, jd, reason):
    return {'result_row': {
        'filename': fits_path.name, 'jd': jd, 'target_detected': False,
        'target_mag': np.nan, 'target_mag_error': np.nan, 'target_x': np.nan, 'target_y': np.nan,
        'num_reference_stars': 0, 'zeropoint': np.nan, 'rms_scatter': np.nan,
        'calibration_mode': None, 'quality_ok': False, 'rejection_reason': reason,
    }, 'ref_rows': []}

progress_bar = widgets.IntProgress(value=0, min=0, max=len(cutouts), description='Plates:')
progress_html = widgets.HTML(value="")
display(widgets.VBox([progress_bar, progress_html]))

start = time.monotonic()
last_render = 0.0
n_processed = 0
for i, f in enumerate(cutouts):
    key = str(f)
    cached = plate_db.get(key)
    if not (cached and cached.get('_version', 0) == PIPELINE_VERSION):
        out = process_plate(f)
        out['_version'] = PIPELINE_VERSION
        plate_db[key] = out
        n_processed += 1

    now = time.monotonic()
    if now - last_render > 0.15 or (i + 1) == len(cutouts):
        elapsed = now - start
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        progress_bar.value = i + 1
        progress_html.value = f"{i+1}/{len(cutouts)} ({rate:.2f} plates/sec)"
        last_render = now

    if (i + 1) % 200 == 0:
        plate_db.sync()

plate_db.sync()
print(f"Processed {n_processed} plates this run, {len(plate_db)} total in cache")

current_keys = {str(f) for f in cutouts}
result_rows, ref_rows_all = [], []
for f_str in plate_db.keys():
    if f_str not in current_keys:
        continue
    out = plate_db[f_str]
    if out is None or out.get('_version') != PIPELINE_VERSION:
        continue
    result_rows.append(out['result_row'])
    ref_rows_all.extend(out['ref_rows'])

print(f"Collected {len(result_rows)} result rows for saving")

if len(result_rows) == 0:
    print("ERROR: No results collected. Something went wrong in the processing loop above.")
else:
    results_df = pd.DataFrame(result_rows).sort_values('jd').reset_index(drop=True)
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Saved {RESULTS_PATH.name} ({len(results_df)} plates)")

    ref_df = pd.DataFrame(ref_rows_all)
    ref_df.to_csv(REF_PHOTOMETRY_PATH, index=False)
    print(f"Saved {REF_PHOTOMETRY_PATH.name} ({len(ref_df)} reference measurements)")

    lc_df = results_df[results_df['quality_ok']].copy()
    lc_df = lc_df.rename(columns={'target_mag': 'magnitude', 'target_mag_error': 'magnitude_error'})
    lc_df = lc_df[['jd', 'magnitude', 'magnitude_error']].dropna()
    lc_df.to_csv(LIGHTCURVE_PATH, index=False)
    print(f"Saved {LIGHTCURVE_PATH.name} ({len(lc_df)} light curve points)")

    print(f"\nCalibration mode breakdown: {results_df['calibration_mode'].value_counts(dropna=False).to_dict()}")
    print(f"\nTo validate: open unit_Test_Generator.ipynb, set USE_SIMULATED_INPUTS=False, run Cell 2")
    print(f"It will auto-discover: {OUTPUT_DIR.name}")

In [ ]:
# 02_B_DASCH_Replica.ipynb - Cell 3: Real V CrA plates
# Now uses solve-field (astrometry.net) for astrometric refinement instead
# of trusting the raw plate header WCS. Header WCS is only a fallback for
# plates where solve-field fails to converge within the time limit.
#
# Two things changed from the previous version:
# 1. solve-field is now given an explicit pixel-scale hint
#    (measured from real cutouts: 1.440 arcsec/px), so it doesn't have
#    to blind-search all scales - this is what was causing 100% timeouts
#    before, combined with having the wrong (4100-series, >1deg) index
#    files installed.
# 2. Star matching now requires the nearest detection to be a CLEAR best
#    match (next-nearest neighbor must be MIN_MATCH_SEPARATION_RATIO x
#    farther away). Without this, a shaky WCS can cross-match a detection
#    to the wrong nearby star and silently poison the light curve - this
#    is the likely cause of the wildly discontinuous points seen in the
#    last real run (mag 10.9 to 16.9 within a few days).

import subprocess
import shelve
import shutil
import time
import tempfile
import traceback
import urllib.request
import urllib.parse
import warnings
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits as astrofits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u
from IPython.display import display
import ipywidgets as widgets

VCRA_DIR = Path("/mnt/c/Users/dapur/Downloads/Other/Research/rcb_Star_Dust_Survey/test_CNN/data/V_CrA")
VCRA_CUTOUTS_DIR = VCRA_DIR / 'cutouts'

RUN_TIMESTAMP = time.strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = VCRA_DIR / '02_B' / f'run_{RUN_TIMESTAMP}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = VCRA_DIR / '02_B' / 'algorithm_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / 'vcra_photometry_results.csv'
LIGHTCURVE_PATH = OUTPUT_DIR / 'vcra_lightcurve.csv'
REF_PHOTOMETRY_PATH = OUTPUT_DIR / 'vcra_reference_photometry.csv'

SUMMARY_DIR = VCRA_DIR / '02_B'
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = SUMMARY_DIR / f'analysis_Summary_{RUN_TIMESTAMP}.txt'

print(f"Output folder: {OUTPUT_DIR}")

MATCH_RADIUS_PX = 5.0
MIN_MATCH_SEPARATION_RATIO = 2.0  # nearest match must be 2x closer than 2nd-nearest

# Distortion refinement (Laycock et al. 2010, Section 3): a raw TAN-equivalent
# solve isn't accurate enough for a tight match, especially away from plate
# center. DASCH's own pipeline does a rough match at 20 arcsec, rejects
# mismatches via a preliminary >1 mag calibration cut, then fits a polynomial
# distortion correction (their TNX step) before the real tight match. We
# replicate that same sequence here instead of matching tightly against the
# raw solve.
ROUGH_MATCH_RADIUS_ARCSEC = 20.0
ROUGH_MATCH_MAG_REJECT = 1.0
MIN_DISTORTION_FIT_STARS = 15
ISOLATION_MIN_SEPARATION = 20
REF_STARS_MAX_PER_PLATE = 150
MIN_REF_STARS = 25
MIN_APASS_STARS_QUERY = 10
LOCAL_HALF_WINDOW = 1.5
MIN_REF_STARS_LOCAL = 8
LOCAL_WINDOW_GROWTH = 1.6
MIN_ABS_LOCAL_SLOPE = 0.3
MAX_CALIBRATION_RMS = 0.235

# Measured from real V_CrA cutouts (835x835 px, 1.440 arcsec/px). +/-10%
# tolerance covers minor scale variation between plates/scans.
PLATE_PIXEL_SCALE_ARCSEC = 1.440
SOLVE_SCALE_LOW = PLATE_PIXEL_SCALE_ARCSEC * 0.90
SOLVE_SCALE_HIGH = PLATE_PIXEL_SCALE_ARCSEC * 1.10
SOLVE_FIELD_TIMEOUT = 12  # seconds, per plate - if it hasn't solved by
# now with a correct scale hint, it isn't going to; waiting longer just
# burns time on failures

PIPELINE_VERSION = 6  # bumped: added median_nearest_dist_px - measures actual WCS error instead of guessing radius
VCRA_TARGET_COORD = SkyCoord(ra=281.884623417 * u.deg, dec=-38.158974417 * u.deg)

# Set to an integer for a fast test run (e.g. 500), or None for the full
# dataset. Spread evenly across the archive (not just the first N by
# filename) so a test run still samples different plate series/eras.
PLATE_LIMIT = 500

_all_cutouts = sorted(VCRA_CUTOUTS_DIR.glob('*.fits'))
if PLATE_LIMIT is not None and PLATE_LIMIT < len(_all_cutouts):
    _step = len(_all_cutouts) / PLATE_LIMIT
    vcra_cutouts = [_all_cutouts[int(i * _step)] for i in range(PLATE_LIMIT)]
else:
    vcra_cutouts = _all_cutouts
print(f"Found {len(_all_cutouts)} V CrA plates total, using {len(vcra_cutouts)} this run "
      f"({'full dataset' if PLATE_LIMIT is None else f'PLATE_LIMIT={PLATE_LIMIT}'})")

plate_db = shelve.open(str(CACHE_DIR / 'plate_db_shelf'), flag='c', writeback=False)

# apass_cache: shelve on this system is SQLite-backed, and SQLite connections
# are hard-tied to the thread that created them - accessing one from a worker
# thread crashes every time, no matter how it's locked. So we load it into a
# plain in-memory dict once (thread-safe under a lock), and persist that dict
# back to the shelve only from the main thread.
_apass_shelf = shelve.open(str(CACHE_DIR / 'apass_cache_shelf'), flag='c', writeback=False)
apass_cache = dict(_apass_shelf)
_apass_shelf.close()
print(f"Loaded {len(plate_db)} cached plate results, {len(apass_cache)} cached APASS field queries")

cache_lock = threading.Lock()  # guards plate_db writes and apass_cache dict access
N_WORKERS = max(1, min(8, (os.cpu_count() or 4)))
print(f"Using {N_WORKERS} concurrent workers")

def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return "calculating..."
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def run_sextractor(fits_path, workdir):
    workdir = Path(workdir)
    catalog_out = workdir / f"{fits_path.stem}.cat"
    cmd = [
        SEXTRACTOR_CMD, str(fits_path),
        '-c', str(EXTRACTOR_CONFIG_FILE),
        '-CATALOG_NAME', str(catalog_out),
        '-PARAMETERS_NAME', str(EXTRACTOR_PARAM_FILE),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if not catalog_out.exists():
        return None, result.stderr
    colnames = []
    with open(catalog_out) as f:
        for line in f:
            if line.startswith('#'):
                colnames.append(line.split()[2])
            else:
                break
    if not colnames:
        return None, "empty catalog"
    df = pd.read_csv(catalog_out, comment='#', sep=r'\s+', names=colnames)
    return df, None

SOLVE_MAX_STARS = 150  # curated star count fed to solve-field

def build_curated_xylist(sources, w, h, workdir, stem):
    """Build a small, clean FITS xylist (brightest, non-saturated, isolated
    stars only) for solve-field to build quads from. Letting solve-field do
    its own source extraction on a real plate picks up hundreds of grain/
    defect detections that swamp the real quads and prevent it from ever
    reaching the solve confidence threshold - this sidesteps that by handing
    it a pre-cleaned list instead."""
    x = sources['X_IMAGE'].values
    y = sources['Y_IMAGE'].values
    mag = sources['MAG_ISO'].values
    flags = sources['FLAGS'].values.astype(int)
    saturated = (flags & 4) != 0

    isolated = filter_isolated(x, y)
    good = isolated & ~saturated & np.isfinite(mag)
    if good.sum() == 0:
        return None

    gx, gy, gmag = x[good], y[good], mag[good]
    order = np.argsort(gmag)  # brightest (lowest mag) first
    order = order[:SOLVE_MAX_STARS]
    gx, gy = gx[order], gy[order]

    xyls_path = Path(workdir) / f"{stem}.xyls"
    col_x = astrofits.Column(name='X', format='E', array=gx)
    col_y = astrofits.Column(name='Y', format='E', array=gy)
    hdu = astrofits.BinTableHDU.from_columns([col_x, col_y])
    hdu.writeto(xyls_path, overwrite=True)
    return xyls_path

def run_solve_field(xyls_path, w, h, workdir, stem):
    """Try to astrometrically solve using a pre-curated star list (not the
    raw FITS - see build_curated_xylist). Returns (WCS or None, solved_bool)."""
    workdir = Path(workdir)
    cmd = [
        'solve-field', str(xyls_path),
        '--width', str(w), '--height', str(h),
        '--dir', str(workdir),
        '--scale-units', 'arcsecperpix',
        '--scale-low', str(SOLVE_SCALE_LOW),
        '--scale-high', str(SOLVE_SCALE_HIGH),
        '--cpulimit', str(SOLVE_FIELD_TIMEOUT),
        '--no-plots', '--overwrite', '--no-verify',
    ]
    try:
        subprocess.run(cmd, capture_output=True, text=True, timeout=SOLVE_FIELD_TIMEOUT + 5)
    except subprocess.TimeoutExpired:
        return None, False

    wcs_path = workdir / f"{xyls_path.stem}.wcs"
    if not wcs_path.exists():
        return None, False
    try:
        solved_header = astrofits.getheader(wcs_path)
        return WCS(solved_header), True
    except Exception:
        return None, False

def fit_distortion_correction(wcs, ra, dec, apass_b, det_x, det_y, det_mag, w, h):
    """Mirrors DASCH's astrometric refinement (Laycock et al. 2010, Sec 3):
    predict catalog pixel positions from the initial (solve-field or header)
    WCS, match loosely (20 arcsec), reject obvious mismatches via a quick
    preliminary magnitude calibration (>1 mag residual = reject, same rule
    the paper uses), then fit a low-order 2D polynomial to the surviving
    (predicted -> observed) position residuals. Returns corrected catalog
    pixel positions for ALL input stars (not just the ones used to fit),
    plus whether refinement actually happened (vs. falling back unrefined
    when there aren't enough stars to fit reliably).
    """
    cat_x0, cat_y0 = wcs.all_world2pix(ra, dec, 0)
    rough_radius_px = ROUGH_MATCH_RADIUS_ARCSEC / PLATE_PIXEL_SCALE_ARCSEC

    if len(det_x) == 0 or len(cat_x0) == 0:
        return cat_x0, cat_y0, False

    matched_i, matched_j = [], []
    for i, (cx, cy) in enumerate(zip(cat_x0, cat_y0)):
        d = np.hypot(det_x - cx, det_y - cy)
        j = np.argmin(d)
        if d[j] <= rough_radius_px:
            matched_i.append(i)
            matched_j.append(j)

    if len(matched_i) < MIN_DISTORTION_FIT_STARS:
        return cat_x0, cat_y0, False

    mi, mj = np.array(matched_i), np.array(matched_j)
    inst, catmag = det_mag[mj], apass_b[mi]
    finite = np.isfinite(inst) & np.isfinite(catmag)
    if finite.sum() < MIN_DISTORTION_FIT_STARS:
        return cat_x0, cat_y0, False

    try:
        prelim_coeffs = np.polyfit(catmag[finite], inst[finite], 2)
    except Exception:
        return cat_x0, cat_y0, False
    resid = inst[finite] - np.polyval(prelim_coeffs, catmag[finite])
    good = np.abs(resid) <= ROUGH_MATCH_MAG_REJECT
    mi_good, mj_good = mi[finite][good], mj[finite][good]

    if len(mi_good) < MIN_DISTORTION_FIT_STARS:
        return cat_x0, cat_y0, False

    px, py = cat_x0[mi_good], cat_y0[mi_good]
    dx, dy = det_x[mj_good] - px, det_y[mj_good] - py
    design = np.column_stack([np.ones_like(px), px, py, px**2, px * py, py**2])
    try:
        coeff_x, *_ = np.linalg.lstsq(design, dx, rcond=None)
        coeff_y, *_ = np.linalg.lstsq(design, dy, rcond=None)
    except Exception:
        return cat_x0, cat_y0, False

    design_full = np.column_stack([np.ones_like(cat_x0), cat_x0, cat_y0,
                                    cat_x0**2, cat_x0 * cat_y0, cat_y0**2])
    cat_x_corr = cat_x0 + design_full @ coeff_x
    cat_y_corr = cat_y0 + design_full @ coeff_y
    return cat_x_corr, cat_y_corr, True

def filter_isolated(x, y, min_sep=ISOLATION_MIN_SEPARATION):
    n = len(x)
    if n < 2:
        return np.ones(n, dtype=bool)
    isolated = np.ones(n, dtype=bool)
    for i in range(n):
        d = np.hypot(x - x[i], y - y[i])
        d[i] = np.inf
        if np.any(d < min_sep):
            isolated[i] = False
    return isolated

def calibrate_global(inst_mags, apass_b, min_stars=MIN_REF_STARS):
    mask = np.isfinite(inst_mags) & np.isfinite(apass_b)
    if mask.sum() < min_stars:
        return None
    x, y = apass_b[mask], inst_mags[mask]
    for _ in range(3):
        try:
            coeffs = np.polyfit(x, y, 2)
            residuals = y - np.polyval(coeffs, x)
            rms = np.sqrt(np.mean(residuals**2))
            good = np.abs(residuals) < 3.0 * rms
            if good.sum() < min_stars:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    if len(x) < min_stars:
        return None
    coeffs = np.polyfit(x, y, 2)
    residuals = y - np.polyval(coeffs, x)
    rms = np.sqrt(np.mean(residuals**2))
    return {'coeffs': coeffs, 'rms': rms, 'n_used': len(x),
            'b_min': float(np.min(x)), 'b_max': float(np.max(x))}

def calibrate_local(inst_mags, apass_b, target_b_estimate,
                     half_window=LOCAL_HALF_WINDOW, min_stars=MIN_REF_STARS_LOCAL,
                     growth=LOCAL_WINDOW_GROWTH):
    mask = np.isfinite(inst_mags) & np.isfinite(apass_b)
    x_all, y_all = apass_b[mask], inst_mags[mask]
    if len(x_all) < min_stars:
        return None
    window = half_window
    local_mask = np.abs(x_all - target_b_estimate) <= window
    for _ in range(5):
        if local_mask.sum() >= min_stars:
            break
        window *= growth
        local_mask = np.abs(x_all - target_b_estimate) <= window
    if local_mask.sum() < min_stars:
        return None
    x, y = x_all[local_mask], y_all[local_mask]
    for _ in range(3):
        try:
            coeffs = np.polyfit(x, y, 1)
            residuals = y - np.polyval(coeffs, x)
            rms = np.sqrt(np.mean(residuals**2))
            good = np.abs(residuals) < 3.0 * rms
            if good.sum() < min_stars:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    if len(x) < min_stars:
        return None
    coeffs = np.polyfit(x, y, 1)
    residuals = y - np.polyval(coeffs, x)
    rms = np.sqrt(np.mean(residuals**2))
    return {'coeffs': coeffs, 'rms': rms, 'n_used': len(x), 'window_used': float(window),
            'b_min': float(np.min(x)), 'b_max': float(np.max(x))}

def invert_calibration(calibration, t_inst_mag):
    coeffs = list(calibration['coeffs'])
    coeffs[-1] -= t_inst_mag
    roots = np.roots(coeffs)
    real_roots = roots[np.abs(roots.imag) < 1e-6].real
    if len(real_roots) == 0:
        return None
    lo, hi = calibration['b_min'], calibration['b_max']
    in_range = real_roots[(real_roots >= lo - 2) & (real_roots <= hi + 2)]
    candidates = in_range if len(in_range) > 0 else real_roots
    return float(min(candidates, key=lambda r: abs(np.polyval(calibration['coeffs'], r) - t_inst_mag)))

def query_apass(ra_c, dec_c, radius_arcsec, max_rows=3000):
    params = {
        '-source': 'II/336/apass9',
        '-c': f"{ra_c} {dec_c}",
        '-c.rs': radius_arcsec,
        '-out': 'RAJ2000,DEJ2000,Bmag,e_Bmag',
        '-out.max': max_rows,
    }
    url = "https://vizier.cds.unistra.fr/viz-bin/asu-tsv?" + urllib.parse.urlencode(params)
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as response:
                return response.read().decode('utf-8')
        except Exception:
            time.sleep(2 ** attempt)
    return None

def parse_apass_tsv(text):
    rows = []
    for line in text.split('\n'):
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('-'):
            continue
        parts = line.split('\t')
        if len(parts) < 4:
            continue
        try:
            ra, dec, bmag, e_bmag = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
            rows.append((ra, dec, bmag, e_bmag))
        except (ValueError, IndexError):
            continue
    if rows and not (0 < rows[0][2] < 25):
        rows = rows[1:]
    return rows

def get_field_catalog(ra_c, dec_c, radius_arcsec):
    key = f"{round(ra_c, 3)}_{round(dec_c, 3)}"
    with cache_lock:
        if key in apass_cache:
            return apass_cache[key]
    # network call happens outside the lock so other threads aren't blocked on it
    raw = query_apass(ra_c, dec_c, radius_arcsec)
    rows = []
    if raw:
        rows = parse_apass_tsv(raw)
        rows = [r for r in rows if 0 < r[2] < 20 and r[3] < 0.5]
    with cache_lock:
        apass_cache[key] = rows  # plain dict write, safe under the lock
    return rows

def get_plate_jd(header):
    if 'JD-OBS' in header:
        return float(header['JD-OBS'])
    if 'MJD-OBS' in header:
        return float(header['MJD-OBS']) + 2400000.5
    for key in ('DATE-OBS', 'DATE'):
        if key in header:
            try:
                return Time(header[key]).jd
            except Exception:
                continue
    return np.nan

def match_unique(det_x, det_y, cx, cy, max_radius=MATCH_RADIUS_PX,
                  min_ratio=MIN_MATCH_SEPARATION_RATIO):
    """Nearest-neighbor match that rejects ambiguous matches: the best
    candidate must be clearly closer than the second-best, or we return
    no match (better to lose a star than mismatch it to a neighbor)."""
    d = np.hypot(det_x - cx, det_y - cy)
    if len(d) == 0:
        return None
    order = np.argsort(d)
    best = order[0]
    if d[best] > max_radius:
        return None
    if len(order) > 1:
        second = order[1]
        if d[second] < d[best] * min_ratio:
            return None  # too ambiguous, could be either star
    return best

def process_plate(fits_path):
    t0 = time.monotonic()
    try:
        plate_rng = np.random.default_rng(abs(hash(fits_path.name)) % (2**32))
        header = astrofits.getheader(fits_path)
        jd = get_plate_jd(header)
        h, w = astrofits.getdata(fits_path).shape

        with tempfile.TemporaryDirectory() as workdir:
            # SExtractor runs FIRST now - solve-field uses its output as a
            # curated star list rather than doing its own noisy extraction.
            sources, err = run_sextractor(fits_path, workdir)
            if sources is None or len(sources) == 0:
                return _empty_result(fits_path, jd, f'SExtractor found nothing ({err})', False)

            xyls_path = build_curated_xylist(sources, w, h, workdir, fits_path.stem)
            wcs, astrometry_solved = (None, False)
            if xyls_path is not None:
                wcs, astrometry_solved = run_solve_field(xyls_path, w, h, workdir, fits_path.stem)
            if wcs is None:
                # Fall back to trusted header WCS if solve-field fails
                try:
                    wcs = WCS(header)
                    astrometry_solved = False
                except Exception:
                    return _empty_result(fits_path, jd, 'No WCS available (solve-field failed, header WCS invalid)')

        det_x = sources['X_IMAGE'].values
        det_y = sources['Y_IMAGE'].values
        det_mag_iso = sources['MAG_ISO'].values
        det_flags = sources['FLAGS'].values.astype(int)
        is_saturated_flag = (det_flags & 4) != 0

        ra_c, dec_c = wcs.all_pix2world(w / 2, h / 2, 0)
        corners_x, corners_y = [0, w - 1, 0, w - 1], [0, 0, h - 1, h - 1]
        corner_ra, corner_dec = wcs.all_pix2world(corners_x, corners_y, 0)
        center = SkyCoord(float(ra_c) * u.deg, float(dec_c) * u.deg)
        corners_sky = SkyCoord(corner_ra * u.deg, corner_dec * u.deg)
        field_radius_arcsec = center.separation(corners_sky).max().arcsec * 1.05

        apass_stars = get_field_catalog(float(ra_c), float(dec_c), field_radius_arcsec)
        if len(apass_stars) < MIN_APASS_STARS_QUERY:
            return _empty_result(fits_path, jd, 'Too few APASS stars in field', astrometry_solved)

        ra = np.array([s[0] for s in apass_stars])
        dec = np.array([s[1] for s in apass_stars])
        apass_b_all = np.array([s[2] for s in apass_stars])
        cat_x, cat_y, distortion_refined = fit_distortion_correction(
            wcs, ra, dec, apass_b_all, det_x, det_y, det_mag_iso, w, h)
        margin = 20
        in_frame = (cat_x > margin) & (cat_x < w - margin) & (cat_y > margin) & (cat_y < h - margin)
        cat_x, cat_y, apass_b_all = cat_x[in_frame], cat_y[in_frame], apass_b_all[in_frame]

        if len(cat_x) > REF_STARS_MAX_PER_PLATE:
            keep = plate_rng.choice(len(cat_x), size=REF_STARS_MAX_PER_PLATE, replace=False)
            cat_x, cat_y, apass_b_all = cat_x[keep], cat_y[keep], apass_b_all[keep]

        n_apass_in_frame = len(cat_x)  # DIAG: catalog stars landing on the frame at all

        # DIAG: for every catalog star, the distance to its single nearest
        # SExtractor detection - NO radius cutoff. This tells us how far off
        # the WCS positions actually are from real data, so we can set
        # MATCH_RADIUS_PX from measurement instead of another guess.
        nearest_dists_px = []
        if len(det_x) > 0:
            for cx, cy in zip(cat_x, cat_y):
                d = np.hypot(det_x - cx, det_y - cy)
                nearest_dists_px.append(float(d.min()))

        matched_inst_mag, matched_apass_b, matched_x, matched_y, matched_sat = [], [], [], [], []
        for cx, cy, ab in zip(cat_x, cat_y, apass_b_all):
            j = match_unique(det_x, det_y, cx, cy)
            if j is not None:
                matched_inst_mag.append(det_mag_iso[j])
                matched_apass_b.append(ab)
                matched_x.append(det_x[j])
                matched_y.append(det_y[j])
                matched_sat.append(is_saturated_flag[j])

        n_matched_raw = len(matched_inst_mag)  # DIAG: survived match_unique (radius + uniqueness)
        median_nearest_dist_px = float(np.median(nearest_dists_px)) if nearest_dists_px else np.nan
        if n_matched_raw == 0:
            return _empty_result(fits_path, jd, 'No SExtractor detections matched catalog stars', astrometry_solved,
                                  distortion_refined, n_apass_in_frame, n_matched_raw,
                                  median_nearest_dist_px=median_nearest_dist_px)

        matched_inst_mag = np.array(matched_inst_mag)
        matched_apass_b = np.array(matched_apass_b)
        matched_x = np.array(matched_x)
        matched_y = np.array(matched_y)
        matched_sat = np.array(matched_sat)

        isolated = filter_isolated(matched_x, matched_y)
        n_isolated = int(isolated.sum())  # DIAG
        n_nonsaturated = int((~matched_sat).sum())  # DIAG
        good_for_fit = isolated & ~matched_sat & np.isfinite(matched_inst_mag)
        n_good_for_fit = int(good_for_fit.sum())  # DIAG: what calibrate_global actually receives

        calibration_global = calibrate_global(matched_inst_mag[good_for_fit], matched_apass_b[good_for_fit])
        n_ref_used = calibration_global['n_used'] if calibration_global else n_good_for_fit

        ref_rows = []
        for i in np.where(good_for_fit)[0]:
            ref_rows.append({'filename': fits_path.name, 'star_id': f'MATCH_{i:04d}',
                              'apass_b_mag': matched_apass_b[i], 'instrumental_mag': matched_inst_mag[i]})

        tx, ty = wcs.all_world2pix(VCRA_TARGET_COORD.ra.deg, VCRA_TARGET_COORD.dec.deg, 0)
        target_detected = False
        target_mag = np.nan
        target_mag_error = np.nan
        calibration_mode = None
        final_rms = np.nan
        final_n_used = 0

        if margin < tx < w - margin and margin < ty < h - margin and calibration_global is not None:
            j = match_unique(det_x, det_y, tx, ty)
            if j is not None:
                t_inst_mag = det_mag_iso[j]
                rough_mag = invert_calibration(calibration_global, t_inst_mag)
                if rough_mag is not None:
                    calibration_local = calibrate_local(matched_inst_mag[good_for_fit],
                                                          matched_apass_b[good_for_fit], rough_mag)
                    if (calibration_local is not None
                            and abs(calibration_local['coeffs'][0]) >= MIN_ABS_LOCAL_SLOPE):
                        local_mag = invert_calibration(calibration_local, t_inst_mag)
                        if local_mag is not None:
                            target_mag = local_mag
                            final_rms = calibration_local['rms']
                            final_n_used = calibration_local['n_used']
                            calibration_mode = 'local'
                    if calibration_mode is None:
                        target_mag = rough_mag
                        final_rms = calibration_global['rms']
                        final_n_used = calibration_global['n_used']
                        calibration_mode = 'global_fallback'
                    target_mag_error = final_rms / np.sqrt(max(final_n_used, 1))
                    target_detected = True

        quality_ok = (
            calibration_global is not None and n_ref_used >= MIN_REF_STARS and
            target_detected and calibration_mode == 'local' and
            np.isfinite(final_rms) and final_rms <= MAX_CALIBRATION_RMS
        )

        rejection_reason = None
        if calibration_global is None:
            rejection_reason = 'Global calibration failed'
        elif n_ref_used < MIN_REF_STARS:
            rejection_reason = f'Too few reference stars ({n_ref_used} < {MIN_REF_STARS})'
        elif not target_detected:
            rejection_reason = 'Target not detected/matched (or match too ambiguous)'
        elif calibration_mode != 'local':
            rejection_reason = 'Local calibration unavailable/unstable'
        elif final_rms > MAX_CALIBRATION_RMS:
            rejection_reason = f'RMS too high ({final_rms:.3f})'

        result_row = {
            'filename': fits_path.name, 'jd': jd, 'target_detected': target_detected,
            'target_mag': target_mag, 'target_mag_error': target_mag_error,
            'target_x': float(tx), 'target_y': float(ty),
            'num_reference_stars': n_ref_used,
            'zeropoint': calibration_global['coeffs'][-1] if calibration_global else np.nan,
            'rms_scatter': final_rms,
            'calibration_mode': calibration_mode, 'quality_ok': quality_ok,
            'rejection_reason': rejection_reason,
            'astrometry_solved': astrometry_solved,
            'distortion_refined': distortion_refined,
            # DIAG columns - trace exactly where the funnel loses stars:
            # apass_in_frame -> matched_raw -> (isolated, nonsaturated) -> good_for_fit -> n_ref_used
            'n_apass_in_frame': n_apass_in_frame,
            'n_matched_raw': n_matched_raw,
            'n_isolated': n_isolated,
            'n_nonsaturated': n_nonsaturated,
            'n_good_for_fit': n_good_for_fit,
            'median_nearest_dist_px': median_nearest_dist_px,
            'process_time_s': time.monotonic() - t0,
        }
        return {'result_row': result_row, 'ref_rows': ref_rows}

    except Exception as e:
        return _empty_result(fits_path, np.nan, str(e), False)

def _empty_result(fits_path, jd, reason, astrometry_solved=False, distortion_refined=False,
                   n_apass_in_frame=0, n_matched_raw=0, n_isolated=0, n_nonsaturated=0, n_good_for_fit=0,
                   median_nearest_dist_px=np.nan):
    return {'result_row': {
        'filename': fits_path.name, 'jd': jd, 'target_detected': False,
        'target_mag': np.nan, 'target_mag_error': np.nan, 'target_x': np.nan, 'target_y': np.nan,
        'num_reference_stars': 0, 'zeropoint': np.nan, 'rms_scatter': np.nan,
        'calibration_mode': None, 'quality_ok': False, 'rejection_reason': reason,
        'astrometry_solved': astrometry_solved, 'distortion_refined': distortion_refined,
        'n_apass_in_frame': n_apass_in_frame, 'n_matched_raw': n_matched_raw,
        'n_isolated': n_isolated, 'n_nonsaturated': n_nonsaturated, 'n_good_for_fit': n_good_for_fit,
        'median_nearest_dist_px': median_nearest_dist_px,
        'process_time_s': np.nan,
    }, 'ref_rows': []}

# --- Progress bar: bar + elapsed + ETA + rate + current plate + solve stats ---
progress_bar = widgets.IntProgress(
    value=0, min=0, max=len(vcra_cutouts),
    layout=widgets.Layout(width='70%', height='20px')
)
progress_bar.style.bar_color = '#3498db'
status_label = widgets.Label(value="")
display(widgets.VBox([progress_bar, status_label]))

run_start_time = time.monotonic()
last_render = 0.0
n_done = 0
n_processed = 0
n_solved = 0
n_solve_failed = 0

# Plates already cached under this pipeline version don't need reprocessing
to_process = []
for f in vcra_cutouts:
    cached = plate_db.get(str(f))
    if cached and cached.get('_version', 0) == PIPELINE_VERSION:
        n_done += 1
    else:
        to_process.append(f)

progress_bar.value = n_done
print(f"{n_done} plates already cached, {len(to_process)} to process this run "
      f"with {N_WORKERS} workers")

def _process_and_tag(f):
    return f, process_plate(f)

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(_process_and_tag, f): f for f in to_process}
    for future in as_completed(futures):
        f, out = future.result()
        out['_version'] = PIPELINE_VERSION
        with cache_lock:
            plate_db[str(f)] = out
        n_done += 1
        n_processed += 1
        if out['result_row'].get('astrometry_solved'):
            n_solved += 1
        else:
            n_solve_failed += 1

        now = time.monotonic()
        if now - last_render > 0.5 or n_done == len(vcra_cutouts):
            elapsed = now - run_start_time
            rate = n_processed / elapsed if elapsed > 0 else 0
            remaining = (len(to_process) - n_processed) / rate if rate > 0 else None
            progress_bar.value = n_done
            status_label.value = (
                f"{n_done}/{len(vcra_cutouts)} | Elapsed {format_eta(elapsed)} | "
                f"ETA {format_eta(remaining)} | {rate:.2f} plates/sec | "
                f"Solved {n_solved}/{n_processed} | Now: {f.name}"
            )
            last_render = now

        if n_processed % 50 == 0:
            with cache_lock:
                plate_db.sync()
                _apass_shelf = shelve.open(str(CACHE_DIR / 'apass_cache_shelf'), flag='c', writeback=False)
                _apass_shelf.update(apass_cache)
                _apass_shelf.sync()
                _apass_shelf.close()

with cache_lock:
    plate_db.sync()
    _apass_shelf = shelve.open(str(CACHE_DIR / 'apass_cache_shelf'), flag='c', writeback=False)
    _apass_shelf.update(apass_cache)
    _apass_shelf.sync()
    _apass_shelf.close()
total_elapsed = time.monotonic() - run_start_time
print(f"Processed {n_processed} plates this run, {len(plate_db)} total in cache")
print(f"solve-field succeeded on {n_solved}/{n_processed} freshly-processed plates "
      f"({n_solved/max(n_processed,1)*100:.1f}%)")
print(f"Total run time: {format_eta(total_elapsed)}")

current_keys = {str(f) for f in vcra_cutouts}
result_rows, ref_rows_all = [], []
for f_str in plate_db.keys():
    if f_str not in current_keys:
        continue
    out = plate_db[f_str]
    if out is None or out.get('_version') != PIPELINE_VERSION:
        continue
    result_rows.append(out['result_row'])
    ref_rows_all.extend(out['ref_rows'])

print(f"Collected {len(result_rows)} result rows for saving")

dasch_jd, dasch_mag, dasch_mag_err = None, None, None
results_df = pd.DataFrame()
lc_df = pd.DataFrame()

if len(result_rows) == 0:
    print("ERROR: No results collected.")
else:
    results_df = pd.DataFrame(result_rows).sort_values('jd').reset_index(drop=True)
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Saved {RESULTS_PATH.name} ({len(results_df)} plates)")

    ref_df = pd.DataFrame(ref_rows_all)
    ref_df.to_csv(REF_PHOTOMETRY_PATH, index=False)
    print(f"Saved {REF_PHOTOMETRY_PATH.name} ({len(ref_df)} reference measurements)")

    lc_df = results_df[results_df['quality_ok']].copy()
    lc_df = lc_df.rename(columns={'target_mag': 'magnitude', 'target_mag_error': 'magnitude_error'})
    lc_df = lc_df[['jd', 'magnitude', 'magnitude_error']].dropna().sort_values('jd').reset_index(drop=True)
    lc_df.to_csv(LIGHTCURVE_PATH, index=False)
    print(f"Saved {LIGHTCURVE_PATH.name} ({len(lc_df)} light curve points)")

    print(f"\nCalibration mode breakdown: {results_df['calibration_mode'].value_counts(dropna=False).to_dict()}")
    print(f"astrometry_solved breakdown: {results_df['astrometry_solved'].value_counts(dropna=False).to_dict()}")
    print(f"distortion_refined breakdown: {results_df['distortion_refined'].value_counts(dropna=False).to_dict()}")
    print("\nTop rejection reasons:")
    print(results_df['rejection_reason'].value_counts(dropna=False).head(10).to_string())

    # DIAG: median funnel counts, for plates that at least got as far as
    # matching (n_matched_raw > 0) - shows exactly where stars are lost:
    # apass_in_frame -> matched_raw -> isolated/nonsaturated -> good_for_fit
    diag_cols = ['n_apass_in_frame', 'n_matched_raw', 'n_isolated', 'n_nonsaturated', 'n_good_for_fit']
    if all(c in results_df.columns for c in diag_cols):
        diag_subset = results_df[results_df['n_matched_raw'] > 0][diag_cols]
        if len(diag_subset) > 0:
            print(f"\nFunnel diagnostics (median, over {len(diag_subset)} plates that matched at least 1 star):")
            print(diag_subset.median().to_string())
            print(f"MIN_REF_STARS threshold is {MIN_REF_STARS} - compare against n_good_for_fit above")

    if 'median_nearest_dist_px' in results_df.columns:
        nd = results_df['median_nearest_dist_px'].dropna()
        if len(nd) > 0:
            print(f"\n*** NEAREST-DETECTION DISTANCE (px), no radius cutoff, n={len(nd)} plates ***")
            print(nd.describe(percentiles=[.25, .5, .75, .9, .95]).to_string())
            print(f"Current MATCH_RADIUS_PX = {MATCH_RADIUS_PX} - this tells us what it actually needs to be.")

    try:
        from daschlab import Session
        sess = Session(str(VCRA_DIR))
        sess.select_target("V CrA")
        sess.select_refcat("apass")
        dlc = sess.lightcurve(0)
        dasch_jd_raw = np.asarray(dlc["time"].jd)
        dasch_mag_raw = np.asarray(dlc["magcal_magdep"].value)
        dasch_err_raw = (np.asarray(dlc["magcal_local_rms"].value)
                          if "magcal_local_rms" in dlc.colnames else np.full_like(dasch_mag_raw, np.nan))
        good = np.isfinite(dasch_jd_raw) & np.isfinite(dasch_mag_raw)
        dasch_jd, dasch_mag, dasch_mag_err = dasch_jd_raw[good], dasch_mag_raw[good], dasch_err_raw[good]
        print(f"DASCH light curve: {len(dasch_jd)} points")
    except Exception as e:
        print(f"Warning: could not fetch DASCH light curve ({e})")

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
    if len(lc_df) > 0:
        ax1.errorbar(lc_df['jd'], lc_df['magnitude'], yerr=lc_df['magnitude_error'],
                     fmt='o', color='#3498db', markersize=4,
                     ecolor='#ef00ff', elinewidth=1.0, capsize=1.5, capthick=0.6,
                     alpha=0.9, markeredgewidth=0)
        ax1.invert_yaxis()
    else:
        ax1.text(0.5, 0.5, 'No quality-passing plates', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_ylabel('B Magnitude')
    ax1.set_title(f'02_B (SExtractor, solve-field WCS) — V CrA ({len(lc_df)} points)')
    ax1.grid(True, alpha=0.3)

    if dasch_jd is not None and len(dasch_jd) > 0:
        ax2.errorbar(dasch_jd, dasch_mag, yerr=dasch_mag_err,
                     fmt='o', color='black', markersize=3,
                     ecolor='#e74c3c', elinewidth=0.6, capsize=1.2, capthick=0.5,
                     alpha=0.85, markeredgewidth=0)
        ax2.invert_yaxis()
        ax2.set_title(f'DASCH Reference (raw, {len(dasch_jd)} points)')
    else:
        ax2.text(0.5, 0.5, 'DASCH light curve unavailable', ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title('DASCH Reference (unavailable)')
    ax2.set_xlabel('Julian Date')
    ax2.set_ylabel('Calibrated Magnitude')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Write the verbose analysis summary file
summary_lines = []
summary_lines.append(f"02_B DASCH Replica Pipeline - Analysis Summary")
summary_lines.append(f"Run timestamp: {RUN_TIMESTAMP}")
summary_lines.append(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"solve-field: scale-low={SOLVE_SCALE_LOW:.3f} scale-high={SOLVE_SCALE_HIGH:.3f} arcsec/px, "
                      f"timeout={SOLVE_FIELD_TIMEOUT}s, falls back to header WCS on failure")
summary_lines.append("")
summary_lines.append("TIMING")
summary_lines.append(f"Total run wall time: {format_eta(total_elapsed)} ({total_elapsed:.1f}s)")
summary_lines.append(f"Plates freshly processed this run: {n_processed}")
if n_processed > 0:
    summary_lines.append(f"Average time per freshly-processed plate: {total_elapsed / n_processed:.2f}s")
    summary_lines.append(f"solve-field success rate: {n_solved}/{n_processed} ({n_solved/n_processed*100:.1f}%)")
summary_lines.append(f"Total plates in dataset: {len(vcra_cutouts)}")
summary_lines.append("")
summary_lines.append("PHOTOMETRY RESULTS")
if len(results_df) > 0:
    summary_lines.append(f"Total plates in results: {len(results_df)}")
    summary_lines.append(f"Calibration mode breakdown: {results_df['calibration_mode'].value_counts(dropna=False).to_dict()}")
    summary_lines.append(f"astrometry_solved breakdown: {results_df['astrometry_solved'].value_counts(dropna=False).to_dict()}")
    summary_lines.append(f"quality_ok = True: {results_df['quality_ok'].sum()} ({results_df['quality_ok'].sum()/len(results_df)*100:.1f}%)")
    rms_vals = results_df['rms_scatter'].dropna()
    if len(rms_vals) > 0:
        summary_lines.append(f"RMS scatter (finite values, n={len(rms_vals)}): "
                              f"median={rms_vals.median():.3f}, mean={rms_vals.mean():.3f}, "
                              f"90th pct={rms_vals.quantile(0.9):.3f}")
    summary_lines.append("Top rejection reasons:")
    for reason, count in results_df['rejection_reason'].value_counts(dropna=False).head(10).items():
        summary_lines.append(f"  {reason}: {count}")
else:
    summary_lines.append("No results collected.")
summary_lines.append("")
summary_lines.append("LIGHT CURVE")
summary_lines.append(f"Final light curve points: {len(lc_df)}")
if len(lc_df) > 0:
    summary_lines.append(f"Magnitude range: {lc_df['magnitude'].min():.2f} to {lc_df['magnitude'].max():.2f}")
    summary_lines.append(f"JD range: {lc_df['jd'].min():.2f} to {lc_df['jd'].max():.2f}")
summary_lines.append("")
summary_lines.append("DASCH COMPARISON")
if dasch_jd is not None:
    summary_lines.append(f"DASCH reference points: {len(dasch_jd)}")
    if len(dasch_jd) > 0:
        summary_lines.append(f"This pipeline's point count as fraction of DASCH: {len(lc_df)/len(dasch_jd)*100:.1f}%")
else:
    summary_lines.append("DASCH light curve unavailable for this run.")
summary_lines.append("")
summary_lines.append("OUTPUT FILES")
summary_lines.append(f"Results: {RESULTS_PATH}")
summary_lines.append(f"Light curve: {LIGHTCURVE_PATH}")
summary_lines.append(f"Reference photometry: {REF_PHOTOMETRY_PATH}")

with open(SUMMARY_PATH, 'w') as f:
    f.write('\n'.join(summary_lines))
print(f"\nSaved analysis summary to {SUMMARY_PATH}")

Output folder: /mnt/c/Users/dapur/Downloads/Other/Research/rcb_Star_Dust_Survey/test_CNN/data/V_CrA/02_B/run_20260810_190858
Found 5402 V CrA plates total, using 500 this run (PLATE_LIMIT=500)
Loaded 5402 cached plate results, 279 cached APASS field queries
Using 8 concurrent workers


0 plates already cached, 500 to process this run with 8 workers
